In [5]:
# ============================================================================
# SECTION 0: SETUP & CONFIGURATION
# ============================================================================

# Standard library imports
import os
import gc
import multiprocessing as mp
import sys
import json
import time
import logging
import warnings
import random
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Union, Any
import pickle

# Data manipulation
import numpy as np
import pandas as pd
import datatable as dt
from datatable import f, by

# Machine Learning
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler

# Deep Learning
import torch

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

# Progress tracking
from tqdm.auto import tqdm
import progressbar

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# ============================================================================
# Global Configuration
# ============================================================================

# Core Parameters
WINDOW_SIZE = 20                    # Sequence window around phosphorylation site
RANDOM_SEED = 42                    # For reproducibility
EXPERIMENT_NAME = "exp_3"           # Experiment identifier
BASE_DIR = f"results/{EXPERIMENT_NAME}"
MAX_SEQUENCE_LENGTH = 5000          # Filter long sequences
BALANCE_CLASSES = True              # 1:1 positive:negative ratio
USE_DATATABLE = True                # Use datatable for speed optimization
BATCH_SIZE = 32                     # For transformer training
GRADIENT_ACCUMULATION_STEPS = 2     # Memory optimization
USE_MIXED_PRECISION = True          # For transformer efficiency

# Set all random seeds for reproducibility
def set_all_seeds(seed: int):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_all_seeds(RANDOM_SEED)

# ============================================================================
# Progress Tracking System
# ============================================================================

class ProgressTracker:
    """Comprehensive progress tracking with checkpoint management"""
    
    def __init__(self, exp_dir: str, auto_cleanup: bool = True):
        self.exp_dir = exp_dir
        self.auto_cleanup = auto_cleanup
        self.progress_file = os.path.join(exp_dir, 'progress_tracker.json')
        self.start_time = datetime.now()
        
        # Create directory structure
        self._create_directories()
        
        # Load or initialize progress
        self.progress = self._load_progress()
        
        # Memory monitoring
        self.memory_threshold = 0.8  # 80% memory usage triggers cleanup
        
    def _create_directories(self):
        """Create all required directories"""
        directories = [
            self.exp_dir,
            os.path.join(self.exp_dir, 'checkpoints'),
            os.path.join(self.exp_dir, 'checkpoints/data_preprocessing'),
            os.path.join(self.exp_dir, 'checkpoints/feature_extraction'),
            os.path.join(self.exp_dir, 'checkpoints/ml_models'),
            os.path.join(self.exp_dir, 'checkpoints/transformers'),
            os.path.join(self.exp_dir, 'checkpoints/ensemble'),
            os.path.join(self.exp_dir, 'ml_models'),
            os.path.join(self.exp_dir, 'transformers'),
            os.path.join(self.exp_dir, 'ensemble'),
            os.path.join(self.exp_dir, 'final_report'),
            os.path.join(self.exp_dir, 'logs'),
            os.path.join(self.exp_dir, 'plots'),
            os.path.join(self.exp_dir, 'plots/data_exploration'),
            os.path.join(self.exp_dir, 'plots/feature_analysis'),
            os.path.join(self.exp_dir, 'plots/ml_models'),
            os.path.join(self.exp_dir, 'plots/transformers'),
            os.path.join(self.exp_dir, 'plots/ensemble'),
            os.path.join(self.exp_dir, 'plots/error_analysis'),
            os.path.join(self.exp_dir, 'plots/final_evaluation'),
            os.path.join(self.exp_dir, 'plots/final_report'),
            os.path.join(self.exp_dir, 'tables'),
            os.path.join(self.exp_dir, 'models')
        ]
        
        for directory in directories:
            os.makedirs(directory, exist_ok=True)
    
    def _load_progress(self) -> Dict:
        """Load progress from file if exists"""
        if os.path.exists(self.progress_file):
            with open(self.progress_file, 'r') as f:
                return json.load(f)
        else:
            return {
                'experiment_start': self.start_time.isoformat(),
                'completed_steps': {},
                'checkpoints': {},
                'metadata': {
                    'experiment_name': EXPERIMENT_NAME,
                    'random_seed': RANDOM_SEED,
                    'window_size': WINDOW_SIZE
                }
            }
    
    def _save_progress(self):
        """Save progress to file"""
        with open(self.progress_file, 'w') as f:
            json.dump(self.progress, f, indent=2, default=str)
    
    def mark_completed(self, step_name: str, metadata: Dict = None, checkpoint_data: Any = None):
        """Mark a step as completed and optionally save checkpoint"""
        completion_time = datetime.now()
        self.progress['completed_steps'][step_name] = {
            'completed_at': completion_time.isoformat(),
            'duration_seconds': (completion_time - self.start_time).total_seconds(),
            'metadata': metadata or {}
        }
        
        if checkpoint_data is not None:
            checkpoint_path = os.path.join(
                self.exp_dir, 'checkpoints', f'{step_name.replace(" ", "_").lower()}.pkl'
            )
            with open(checkpoint_path, 'wb') as f:
                pickle.dump(checkpoint_data, f, protocol=4)
            self.progress['checkpoints'][step_name] = checkpoint_path
        
        self._save_progress()
        
        # Check memory and cleanup if needed
        if self.auto_cleanup:
            self._check_memory_usage()
    
    def is_completed(self, step_name: str) -> bool:
        """Check if a step is already completed"""
        return step_name in self.progress['completed_steps']
    
    def resume_from_checkpoint(self, step_name: str) -> Any:
        """Resume from a checkpoint if exists"""
        if step_name in self.progress['checkpoints']:
            checkpoint_path = self.progress['checkpoints'][step_name]
            if os.path.exists(checkpoint_path):
                with open(checkpoint_path, 'rb') as f:
                    return pickle.load(f)
        return None
    
    def get_progress_summary(self) -> Dict:
        """Get summary of progress"""
        total_steps = 10  # Total number of major sections
        completed_steps = len(self.progress['completed_steps'])
        
        return {
            'total_steps': total_steps,
            'completed_steps': completed_steps,
            'percentage': (completed_steps / total_steps) * 100,
            'elapsed_time': str(datetime.now() - self.start_time),
            'completed': list(self.progress['completed_steps'].keys())
        }
    
    def get_memory_usage(self) -> Dict:
        """Get current memory usage"""
        try:
            import psutil
            process = psutil.Process(os.getpid())
            memory_info = process.memory_info()
            return {
                'rss_mb': memory_info.rss / (1024 * 1024),
                'vms_mb': memory_info.vms / (1024 * 1024),
                'percent': process.memory_percent()
            }
        except ImportError:
            return {'rss_mb': 0, 'vms_mb': 0, 'percent': 0}
    
    def _check_memory_usage(self):
        """Check memory usage and trigger cleanup if needed"""
        memory = self.get_memory_usage()
        if memory['percent'] > self.memory_threshold * 100:
            self.trigger_cleanup()
    
    def trigger_cleanup(self):
        """Trigger memory cleanup"""
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    def force_retrain(self, step_name: str):
        """Force retrain by removing a completed step"""
        if step_name in self.progress['completed_steps']:
            del self.progress['completed_steps'][step_name]
        if step_name in self.progress['checkpoints']:
            checkpoint_path = self.progress['checkpoints'][step_name]
            if os.path.exists(checkpoint_path):
                os.remove(checkpoint_path)
            del self.progress['checkpoints'][step_name]
        self._save_progress()
    
    def export_progress_report(self) -> str:
        """Export detailed progress report"""
        report = f"""
Phosphorylation Prediction Experiment Progress Report
=====================================================
Experiment: {EXPERIMENT_NAME}
Started: {self.progress['experiment_start']}
Current Time: {datetime.now().isoformat()}
Elapsed: {datetime.now() - self.start_time}

Progress Summary:
-----------------
"""
        summary = self.get_progress_summary()
        report += f"Completed: {summary['completed_steps']}/{summary['total_steps']} steps ({summary['percentage']:.1f}%)\n\n"
        
        report += "Completed Steps:\n"
        for step, info in self.progress['completed_steps'].items():
            report += f"- {step}: {info['completed_at']} (Duration: {info['duration_seconds']:.1f}s)\n"
        
        report += f"\nMemory Usage:\n"
        memory = self.get_memory_usage()
        report += f"- RSS: {memory['rss_mb']:.1f} MB\n"
        report += f"- VMS: {memory['vms_mb']:.1f} MB\n"
        report += f"- Percent: {memory['percent']:.1f}%\n"
        
        return report

# ============================================================================
# Logging Setup
# ============================================================================

def setup_logging(log_dir: str):
    """Setup comprehensive logging"""
    log_file = os.path.join(log_dir, 'experiment.log')
    
    # Configure logging
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler(sys.stdout)
        ]
    )
    
    logger = logging.getLogger(__name__)
    logger.info("="*80)
    logger.info(f"Phosphorylation Prediction Experiment: {EXPERIMENT_NAME}")
    logger.info(f"Started at: {datetime.now()}")
    logger.info("="*80)
    
    return logger


# ============================================================================
# Environment Information
# ============================================================================

def log_environment_info(logger):
    """Log complete environment information"""
    logger.info("\nEnvironment Information:")
    logger.info(f"Python version: {sys.version}")
    logger.info(f"NumPy version: {np.__version__}")
    logger.info(f"Pandas version: {pd.__version__}")
    logger.info(f"PyTorch version: {torch.__version__}")
    
    # GPU information
    if torch.cuda.is_available():
        logger.info(f"CUDA available: Yes")
        logger.info(f"CUDA version: {torch.version.cuda}")
        logger.info(f"GPU count: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            logger.info(f"GPU {i}: {torch.cuda.get_device_name(i)}")
            logger.info(f"GPU {i} Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
    else:
        logger.info("CUDA available: No (CPU mode)")
    
    # Memory information
    try:
        import psutil
        memory = psutil.virtual_memory()
        logger.info(f"Total RAM: {memory.total / 1e9:.1f} GB")
        logger.info(f"Available RAM: {memory.available / 1e9:.1f} GB")
    except ImportError:
        logger.info("psutil not available for memory information")


# ============================================================================
# Configuration Export
# ============================================================================

def export_configuration(exp_dir: str):
    """Export complete experiment configuration"""
    config = {
        'experiment': {
            'name': EXPERIMENT_NAME,
            'base_dir': BASE_DIR,
            'created_at': datetime.now().isoformat()
        },
        'data': {
            'window_size': WINDOW_SIZE,
            'max_sequence_length': MAX_SEQUENCE_LENGTH,
            'balance_classes': BALANCE_CLASSES,
            'use_datatable': USE_DATATABLE
        },
        'training': {
            'random_seed': RANDOM_SEED,
            'batch_size': BATCH_SIZE,
            'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
            'use_mixed_precision': USE_MIXED_PRECISION
        },
        'environment': {
            'python_version': sys.version,
            'numpy_version': np.__version__,
            'pandas_version': pd.__version__,
            'torch_version': torch.__version__,
            'cuda_available': torch.cuda.is_available(),
            'gpu_count': torch.cuda.device_count() if torch.cuda.is_available() else 0
        }
    }
    
    config_file = os.path.join(exp_dir, 'experiment_config.yaml')
    with open(config_file, 'w') as f:
        json.dump(config, f, indent=2, default=str)
    
    return config


# ============================================================================
# Initialize Everything
# ============================================================================

print("Initializing Phosphorylation Prediction Experiment...")
print(f"Experiment Name: {EXPERIMENT_NAME}")
print(f"Base Directory: {BASE_DIR}")

# Initialize progress tracker
progress_tracker = ProgressTracker(BASE_DIR)

# Setup logging
logger = setup_logging(os.path.join(BASE_DIR, 'logs'))

# Log environment information
log_environment_info(logger)

# Export configuration
config = export_configuration(BASE_DIR)
logger.info(f"Configuration exported to: {os.path.join(BASE_DIR, 'experiment_config.yaml')}")

# Display progress summary
summary = progress_tracker.get_progress_summary()
print(f"\nProgress: {summary['completed_steps']}/{summary['total_steps']} steps completed ({summary['percentage']:.1f}%)")
if summary['completed_steps'] > 0:
    print("Completed steps:", ", ".join(summary['completed']))

print("\nSetup completed successfully!")
print("="*80)

Initializing Phosphorylation Prediction Experiment...
Experiment Name: exp_3
Base Directory: results/exp_3
2025-07-14 13:14:49,534 - __main__ - INFO - ================================================================================
2025-07-14 13:14:49,536 - __main__ - INFO - Phosphorylation Prediction Experiment: exp_3
2025-07-14 13:14:49,536 - __main__ - INFO - Started at: 2025-07-14 13:14:49.536050
2025-07-14 13:14:49,537 - __main__ - INFO - ================================================================================
2025-07-14 13:14:49,538 - __main__ - INFO - 
Environment Information:
2025-07-14 13:14:49,539 - __main__ - INFO - Python version: 3.9.21 (main, Dec 11 2024, 16:35:24) [MSC v.1929 64 bit (AMD64)]
2025-07-14 13:14:49,540 - __main__ - INFO - NumPy version: 1.26.4
2025-07-14 13:14:49,542 - __main__ - INFO - Pandas version: 2.2.3
2025-07-14 13:14:49,543 - __main__ - INFO - PyTorch version: 2.5.1+cu121
2025-07-14 13:14:49,544 - __main__ - INFO - CUDA available: Yes
2025-07

In [9]:
# ============================================================================
# COMPLETE FIXED ORACLE ANALYSIS WITHOUT V2
# Based on reference code, properly loading ALL test predictions
# ============================================================================

print("\n" + "="*80)
print("🔧 COMPLETE FIXED ORACLE ANALYSIS WITHOUT TRANSFORMER V2")
print("="*80)
print("🎯 Goal: Complete oracle analysis with ALL 8 models (5 ML + 3 Transformers)")
print("🔧 Fix: Using reference code approach to load ML test predictions")
print("📊 Analysis: Proper comparison without transformer_v2")
print("="*80)

import numpy as np
import pandas as pd
import os
import pickle
import sys
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                           f1_score, roc_auc_score, matthews_corrcoef)
import json

# ============================================================================
# CONFIGURATION
# ============================================================================

BASE_DIR = 'results/exp_3'
TRAINING_PRED_DIR = f'{BASE_DIR}/ensemble/training_predictions'
META_LEARNING_DIR = f'{BASE_DIR}/meta_learning'

os.makedirs(META_LEARNING_DIR, exist_ok=True)

# Model files (same as reference, but we'll exclude V2)
MODEL_FILES = {
    # ML Models
    'ml_aac': 'ml_aac_training_predictions.csv',
    'ml_binary': 'ml_binary_training_predictions.csv', 
    'ml_dpc': 'ml_dpc_training_predictions.csv',
    'ml_physicochemical': 'ml_physicochemical_training_predictions.csv',
    'ml_tpc': 'ml_tpc_training_predictions.csv',
    
    # Transformer Models
    'transformer_v1': 'transformer_transformer_v1_20250703_112558_training_predictions.csv',
    'transformer_v2': 'transformer_transformer_v2_20250704_102324_training_predictions.csv',  # Will exclude
    'transformer_v3': 'transformer_transformer_v3_20250709_175607_training_predictions.csv',
    'transformer_v4': 'transformer_transformer_v4_20250710_095513_training_predictions.csv'
}

EXCLUDED_MODELS = ['transformer_v2']
print(f"🚫 Excluding models: {EXCLUDED_MODELS}")

# ============================================================================
# 1. LOAD TRAINING PREDICTIONS (EXCLUDING V2)
# ============================================================================

print("\n1. Loading Training Predictions (Excluding V2)")
print("-" * 40)

training_predictions = {}
training_prediction_matrix = []
model_names = []
training_true_labels = None

for model_name, filename in MODEL_FILES.items():
    # Skip excluded models
    if model_name in EXCLUDED_MODELS:
        print(f"   🚫 Skipping {model_name} (excluded)")
        continue
        
    file_path = os.path.join(TRAINING_PRED_DIR, filename)
    
    if os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            print(f"   ✓ {model_name}: {len(df)} samples")
            
            training_predictions[model_name] = {
                'probabilities': df['probability'].values,
                'predictions': (df['probability'] > 0.5).astype(int),
                'true_labels': df['true_label'].values
            }
            
            training_prediction_matrix.append(df['probability'].values)
            model_names.append(model_name)
            
            if training_true_labels is None:
                training_true_labels = df['true_label'].values
                
        except Exception as e:
            print(f"   ❌ Error loading {model_name}: {e}")
    else:
        print(f"   ⚠️ File not found: {filename}")

training_prediction_matrix = np.column_stack(training_prediction_matrix)
print(f"\n📊 Training data without V2:")
print(f"   Samples: {len(training_true_labels)}")
print(f"   Models: {len(model_names)} (was 9, now {len(model_names)})")
print(f"   Available models: {model_names}")

# ============================================================================
# 2. TRAINING ORACLE ANALYSIS
# ============================================================================

print("\n2. Oracle Selection - Training Data (Without V2)")
print("-" * 40)

def calculate_oracle_performance(prediction_matrix, true_labels, model_names, data_type="Training"):
    """Calculate oracle (perfect selection) performance"""
    
    n_samples, n_models = prediction_matrix.shape
    oracle_predictions = []
    oracle_probabilities = []
    selected_models = []
    
    print(f"🔮 Calculating oracle selection for {n_samples} {data_type.lower()} samples...")
    print(f"   Using {n_models} models (excluding transformer_v2)")
    
    for i in range(n_samples):
        true_label = true_labels[i]
        sample_predictions = prediction_matrix[i]
        
        if true_label == 1:
            best_model_idx = np.argmax(sample_predictions)
        else:
            best_model_idx = np.argmin(sample_predictions)
        
        oracle_probabilities.append(sample_predictions[best_model_idx])
        oracle_predictions.append(1 if sample_predictions[best_model_idx] > 0.5 else 0)
        selected_models.append(best_model_idx)
    
    oracle_predictions = np.array(oracle_predictions)
    oracle_probabilities = np.array(oracle_probabilities)
    selected_models = np.array(selected_models)
    
    metrics = {
        'accuracy': accuracy_score(true_labels, oracle_predictions),
        'precision': precision_score(true_labels, oracle_predictions, zero_division=0),
        'recall': recall_score(true_labels, oracle_predictions, zero_division=0),
        'f1': f1_score(true_labels, oracle_predictions, zero_division=0),
        'auc': roc_auc_score(true_labels, oracle_probabilities),
        'mcc': matthews_corrcoef(true_labels, oracle_predictions)
    }
    
    unique_models, counts = np.unique(selected_models, return_counts=True)
    selection_dist = {}
    for model_idx, count in zip(unique_models, counts):
        if model_idx < len(model_names):
            selection_dist[model_names[model_idx]] = {
                'count': int(count),
                'percentage': float(count / n_samples * 100)
            }
    
    return {
        'metrics': metrics,
        'oracle_predictions': oracle_predictions,
        'oracle_probabilities': oracle_probabilities,
        'selected_models': selected_models,
        'selection_distribution': selection_dist
    }

# Calculate training oracle
training_oracle_no_v2 = calculate_oracle_performance(
    training_prediction_matrix, 
    training_true_labels, 
    model_names, 
    "Training (No V2)"
)

print(f"🎯 Training Oracle Performance (WITHOUT V2):")
for metric, value in training_oracle_no_v2['metrics'].items():
    print(f"   {metric.upper()}: {value:.4f}")

# ============================================================================
# 3. LOAD TEST PREDICTIONS USING REFERENCE CODE APPROACH
# ============================================================================

print("\n3. Loading Test Predictions - Reference Code Approach")
print("-" * 40)

# Try to load progress tracker (exactly like reference code)
try:
    # First, try to create progress tracker
    sys.path.append('.')
    from Section0_code import ProgressTracker
    progress_tracker = ProgressTracker(BASE_DIR)
    print("✅ Progress tracker loaded successfully")
    
    # Load required checkpoints (exactly like Section 8)
    required_checkpoints = ['data_loading', 'data_splitting', 'ml_models_enhanced']
    loaded_data = {}

    print("📥 Loading core checkpoints...")
    for checkpoint_name in required_checkpoints:
        try:
            checkpoint_data = progress_tracker.resume_from_checkpoint(checkpoint_name)
            if checkpoint_data:
                loaded_data[checkpoint_name] = checkpoint_data
                print(f"   ✓ Loaded {checkpoint_name} checkpoint")
            else:
                print(f"   ⚠️ {checkpoint_name} checkpoint not found")
        except Exception as e:
            print(f"   ❌ Error loading {checkpoint_name}: {e}")
            
except Exception as e:
    print(f"⚠️ Could not load progress tracker: {e}")
    print("Will try manual loading...")
    
    # Manual loading approach
    loaded_data = {}
    checkpoints_dir = f'{BASE_DIR}/checkpoints'
    
    for checkpoint_name in ['data_loading', 'data_splitting', 'ml_models_enhanced']:
        checkpoint_file = os.path.join(checkpoints_dir, f'{checkpoint_name}.pkl')
        if os.path.exists(checkpoint_file):
            try:
                with open(checkpoint_file, 'rb') as f:
                    loaded_data[checkpoint_name] = pickle.load(f)
                print(f"   ✓ Manually loaded {checkpoint_name}")
            except Exception as e:
                print(f"   ❌ Error manually loading {checkpoint_name}: {e}")

# Extract test data
test_true_labels = None
if 'data_loading' in loaded_data and 'data_splitting' in loaded_data:
    df_final = loaded_data['data_loading']['df_final']
    test_indices = loaded_data['data_splitting']['test_indices']
    
    test_df = df_final.iloc[test_indices]
    test_true_labels = test_df['target'].values
    
    print(f"✅ Test data extracted:")
    print(f"   Test samples: {len(test_indices)}")
    print(f"   Test labels - Positive: {sum(test_true_labels)}, Negative: {len(test_true_labels) - sum(test_true_labels)}")

# ============================================================================
# 4. LOAD ML TEST PREDICTIONS (REFERENCE CODE APPROACH)
# ============================================================================

print("\n4. Loading ML Test Predictions from Checkpoint")
print("-" * 40)

test_predictions = {}

# Load ML test predictions from checkpoint (exactly like reference code)
if 'ml_models_enhanced' in loaded_data:
    ml_data = loaded_data['ml_models_enhanced']
    
    if 'test_predictions' in ml_data:
        ml_test_preds = ml_data['test_predictions']
        print(f"   ✅ Found ML test predictions for: {list(ml_test_preds.keys())}")
        
        for feature_type, pred_data in ml_test_preds.items():
            model_name = f'ml_{feature_type}'
            
            # Skip excluded models
            if model_name in EXCLUDED_MODELS:
                print(f"   🚫 Skipping {model_name} (excluded)")
                continue
            
            if 'pred_proba' in pred_data:
                test_predictions[model_name] = pred_data['pred_proba']
                print(f"   ✓ {model_name}: {len(pred_data['pred_proba'])} test samples")
    else:
        print("   ⚠️ No 'test_predictions' key found in ml_models_enhanced")
else:
    print("   ⚠️ ml_models_enhanced checkpoint not available")

# ============================================================================
# 5. LOAD TRANSFORMER TEST PREDICTIONS (REFERENCE CODE APPROACH)
# ============================================================================

print("\n5. Loading Transformer Test Predictions")
print("-" * 40)

# Load transformer test predictions (exactly like reference code)
transformer_dir = f'{BASE_DIR}/transformers'
if os.path.exists(transformer_dir):
    print(f"   📁 Searching in: {transformer_dir}")
    
    # Map model names to likely directory patterns (reference code approach)
    transformer_patterns = {
        'transformer_v1': ['transformer_v1_20250703_112558', 'transformer_v1'],
        'transformer_v2': ['transformer_v2_20250704_102324', 'transformer_v2'], 
        'transformer_v3': ['transformer_v3_20250709_175607', 'transformer_v3'],
        'transformer_v4': ['transformer_v4_20250710_095513', 'transformer_v4']
    }
    
    for model_name, patterns in transformer_patterns.items():
        # Skip excluded models
        if model_name in EXCLUDED_MODELS:
            print(f"   🚫 Skipping {model_name} (excluded)")
            continue
            
        found = False
        for pattern in patterns:
            for subdir in os.listdir(transformer_dir):
                if pattern in subdir:
                    subdir_path = os.path.join(transformer_dir, subdir)
                    if os.path.isdir(subdir_path):
                        pred_file = os.path.join(subdir_path, 'predictions', 'test_predictions.csv')
                        if os.path.exists(pred_file):
                            try:
                                df = pd.read_csv(pred_file)
                                if 'prediction_prob' in df.columns:
                                    test_predictions[model_name] = df['prediction_prob'].values
                                    print(f"   ✓ {model_name}: {len(df)} test samples")
                                    found = True
                                    break
                                elif 'probability' in df.columns:
                                    test_predictions[model_name] = df['probability'].values
                                    print(f"   ✓ {model_name}: {len(df)} test samples")
                                    found = True
                                    break
                            except Exception as e:
                                print(f"   ❌ Error loading {model_name}: {e}")
                if found:
                    break
            if found:
                break

# ============================================================================
# 6. CREATE COMPLETE TEST PREDICTION MATRIX
# ============================================================================

print("\n6. Creating Complete Test Prediction Matrix (Without V2)")
print("-" * 40)

# Create test prediction matrix in same order as training
test_prediction_matrix = []
available_test_models = []

print(f"📊 Combining test predictions in training order:")
for model_name in model_names:  # Use training model order (already excludes V2)
    if model_name in test_predictions:
        test_prediction_matrix.append(test_predictions[model_name])
        available_test_models.append(model_name)
        print(f"   ✓ {model_name}: Available for oracle analysis")
    else:
        print(f"   ❌ {model_name}: Missing test predictions")

if len(test_prediction_matrix) > 0:
    test_prediction_matrix = np.column_stack(test_prediction_matrix)
    
    print(f"\n📊 COMPLETE test data without V2:")
    print(f"   Available models: {len(available_test_models)} out of {len(model_names)}")
    print(f"   Prediction matrix shape: {test_prediction_matrix.shape}")
    print(f"   Models included: {available_test_models}")
    
    if test_true_labels is not None:
        print(f"   Test samples: {len(test_true_labels)}")
        
        # Verify alignment
        if len(test_true_labels) != test_prediction_matrix.shape[0]:
            min_len = min(len(test_true_labels), test_prediction_matrix.shape[0])
            test_true_labels = test_true_labels[:min_len]
            test_prediction_matrix = test_prediction_matrix[:min_len]
            print(f"   🔧 Truncated both to {min_len} samples for alignment")
        
        # Count ML vs Transformer models in test
        test_ml_count = len([m for m in available_test_models if m.startswith('ml_')])
        test_transformer_count = len([m for m in available_test_models if m.startswith('transformer_')])
        print(f"   📊 Test models: {test_ml_count} ML + {test_transformer_count} Transformers")
        
        # Calculate complete test oracle
        test_oracle_no_v2 = calculate_oracle_performance(
            test_prediction_matrix, 
            test_true_labels, 
            available_test_models, 
            "Test (No V2 - Complete)"
        )
        
        print(f"\n🎯 COMPLETE Test Oracle Performance (WITHOUT V2):")
        for metric, value in test_oracle_no_v2['metrics'].items():
            print(f"   {metric.upper()}: {value:.4f}")
    else:
        print("   ❌ No test labels available")
        test_oracle_no_v2 = None
else:
    print("   ❌ No test predictions available")
    test_oracle_no_v2 = None

# ============================================================================
# 7. COMPLETE BASELINE COMPARISON
# ============================================================================

print("\n7. Complete Baseline Comparison")
print("-" * 40)

def calculate_ensemble_baseline(prediction_matrix, true_labels):
    """Calculate simple average ensemble performance"""
    avg_predictions = np.mean(prediction_matrix, axis=1)
    avg_binary = (avg_predictions > 0.5).astype(int)
    
    return {
        'accuracy': accuracy_score(true_labels, avg_binary),
        'precision': precision_score(true_labels, avg_binary, zero_division=0),
        'recall': recall_score(true_labels, avg_binary, zero_division=0),
        'f1': f1_score(true_labels, avg_binary, zero_division=0),
        'auc': roc_auc_score(true_labels, avg_predictions),
        'mcc': matthews_corrcoef(true_labels, avg_binary)
    }

# Training baseline
training_baseline_no_v2 = calculate_ensemble_baseline(training_prediction_matrix, training_true_labels)

print(f"📊 TRAINING SET COMPARISON (Without V2 - {len(model_names)} models):")
print(f"   Simple Average Ensemble:")
for metric, value in training_baseline_no_v2.items():
    print(f"     {metric.upper()}: {value:.4f}")

print(f"\n   Oracle Selection:")
for metric, value in training_oracle_no_v2['metrics'].items():
    improvement = value - training_baseline_no_v2[metric]
    print(f"     {metric.upper()}: {value:.4f} ({improvement:+.4f})")

# Test baseline
if test_oracle_no_v2 is not None:
    test_baseline_no_v2 = calculate_ensemble_baseline(test_prediction_matrix, test_true_labels)
    
    print(f"\n📊 TEST SET COMPARISON (Without V2 - {len(available_test_models)} models):")
    print(f"   Simple Average Ensemble:")
    for metric, value in test_baseline_no_v2.items():
        print(f"     {metric.upper()}: {value:.4f}")
    
    print(f"\n   Oracle Selection:")
    for metric, value in test_oracle_no_v2['metrics'].items():
        improvement = value - test_baseline_no_v2[metric]
        print(f"     {metric.upper()}: {value:.4f} ({improvement:+.4f})")

# ============================================================================
# 8. DETAILED MODEL SELECTION ANALYSIS
# ============================================================================

print("\n8. Detailed Model Selection Analysis (Complete)")
print("-" * 40)

# Training analysis
print("🔍 Training Set - Model Selection Distribution (No V2):")
for model_name, stats in training_oracle_no_v2['selection_distribution'].items():
    model_type = "ML" if model_name.startswith('ml_') else "Transformer"
    print(f"   {model_name} ({model_type}): {stats['count']} times ({stats['percentage']:.1f}%)")

training_ml_selections = sum(stats['count'] for model, stats in training_oracle_no_v2['selection_distribution'].items() 
                           if model.startswith('ml_'))
training_transformer_selections = sum(stats['count'] for model, stats in training_oracle_no_v2['selection_distribution'].items() 
                                    if model.startswith('transformer_'))
training_total = len(training_true_labels)

print(f"\n📊 Training Set - Model Type Distribution (No V2):")
print(f"   ML Models: {training_ml_selections} ({training_ml_selections/training_total*100:.1f}%)")
print(f"   Transformers: {training_transformer_selections} ({training_transformer_selections/training_total*100:.1f}%)")

if test_oracle_no_v2 is not None:
    print(f"\n🔍 Test Set - Model Selection Distribution (No V2 - COMPLETE):")
    for model_name, stats in test_oracle_no_v2['selection_distribution'].items():
        model_type = "ML" if model_name.startswith('ml_') else "Transformer"
        print(f"   {model_name} ({model_type}): {stats['count']} times ({stats['percentage']:.1f}%)")
    
    test_ml_selections = sum(stats['count'] for model, stats in test_oracle_no_v2['selection_distribution'].items() 
                           if model.startswith('ml_'))
    test_transformer_selections = sum(stats['count'] for model, stats in test_oracle_no_v2['selection_distribution'].items() 
                                    if model.startswith('transformer_'))
    test_total = len(test_true_labels)
    
    print(f"\n📊 Test Set - Model Type Distribution (No V2 - COMPLETE):")
    print(f"   ML Models: {test_ml_selections} ({test_ml_selections/test_total*100:.1f}%)")
    print(f"   Transformers: {test_transformer_selections} ({test_transformer_selections/test_total*100:.1f}%)")

# ============================================================================
# 9. COMPARISON WITH ORIGINAL RESULTS
# ============================================================================

print("\n9. Comparison: WITH vs WITHOUT V2 (COMPLETE)")
print("-" * 40)

print(f"🔄 FINAL COMPARISON ANALYSIS:")

print(f"\n📊 TRAINING SET:")
print(f"   With V2 (9 models):")
print(f"     • Oracle F1: 0.9948")
print(f"     • Baseline F1: 0.9187") 
print(f"     • transformer_v2 selections: 44.0%")

print(f"\n   Without V2 ({len(model_names)} models):")
print(f"     • Oracle F1: {training_oracle_no_v2['metrics']['f1']:.4f}")
print(f"     • Baseline F1: {training_baseline_no_v2['f1']:.4f}")

# Find top training model
top_training_model = max(training_oracle_no_v2['selection_distribution'].items(), 
                        key=lambda x: x[1]['count'])
print(f"     • Top model: {top_training_model[0]} ({top_training_model[1]['percentage']:.1f}%)")

if test_oracle_no_v2 is not None:
    print(f"\n📊 TEST SET:")
    print(f"   With V2 (9 models):")
    print(f"     • Oracle F1: 0.9530")
    print(f"     • Baseline F1: 0.8118")
    print(f"     • transformer_v2 selections: 39.2%")
    
    print(f"\n   Without V2 ({len(available_test_models)} models):")
    print(f"     • Oracle F1: {test_oracle_no_v2['metrics']['f1']:.4f}")
    print(f"     • Baseline F1: {test_baseline_no_v2['f1']:.4f}")
    
    # Find top test model
    top_test_model = max(test_oracle_no_v2['selection_distribution'].items(), 
                        key=lambda x: x[1]['count'])
    print(f"     • Top model: {top_test_model[0]} ({top_test_model[1]['percentage']:.1f}%)")
    
    # Calculate impact
    f1_drop = 0.9530 - test_oracle_no_v2['metrics']['f1']
    baseline_drop = 0.8118 - test_baseline_no_v2['f1']
    
    print(f"\n🎯 IMPACT OF REMOVING V2:")
    print(f"   Oracle F1 drop: {f1_drop:.4f} ({f1_drop/0.9530*100:.1f}%)")
    print(f"   Baseline F1 drop: {baseline_drop:.4f} ({baseline_drop/0.8118*100:.1f}%)")

# ============================================================================
# 10. META-LEARNER IMPLICATIONS (COMPLETE)
# ============================================================================

print("\n10. Meta-Learner Implications (Complete Analysis)")
print("-" * 40)

if test_oracle_no_v2 is not None:
    meta_learner_f1 = 0.7655  # From your results
    baseline_with_v2 = 0.8131
    baseline_without_v2 = test_baseline_no_v2['f1']
    oracle_without_v2 = test_oracle_no_v2['metrics']['f1']
    gap_without_v2 = oracle_without_v2 - baseline_without_v2
    
    print(f"💡 COMPLETE META-LEARNER INSIGHTS:")
    
    print(f"\n📊 Performance Ladder (COMPLETE):")
    print(f"   Meta-learner (with V2): {meta_learner_f1:.4f}")
    print(f"   Baseline (without V2): {baseline_without_v2:.4f}")
    print(f"   Baseline (with V2): {baseline_with_v2:.4f}")
    print(f"   Oracle (without V2): {oracle_without_v2:.4f}")
    print(f"   Oracle (with V2): 0.9530")
    
    if meta_learner_f1 < baseline_without_v2:
        print(f"\n🚨 CRITICAL FINDING:")
        print(f"   Meta-learner STILL performs worse than baseline without V2!")
        print(f"   Gap: {meta_learner_f1 - baseline_without_v2:.4f} F1 points")
        print(f"   💡 The issue is NOT just V2 dominance - deeper problems exist!")
    else:
        print(f"\n✅ Meta-learner beats baseline without V2")
    
    print(f"\n🎯 Meta-learning potential without V2: {gap_without_v2:.4f} F1 points")
    
    if gap_without_v2 > 0.05:
        print(f"   🚀 Still HIGH potential for meta-learning!")
    elif gap_without_v2 > 0.02:
        print(f"   📈 MODERATE potential for meta-learning")
    else:
        print(f"   ⚠️ LIMITED potential - models well-calibrated without V2")
    
    # Check model balance
    max_selection_pct = max(stats['percentage'] for stats in test_oracle_no_v2['selection_distribution'].values())
    print(f"\n📊 Model selection balance without V2:")
    print(f"   Max single model selection: {max_selection_pct:.1f}% (was 39.2% with V2)")
    
    if max_selection_pct < 35:
        print(f"   ✅ More balanced selection - easier for meta-learner!")
    else:
        print(f"   ⚠️ Still some dominance present")

# ============================================================================
# 11. FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("🔮 COMPLETE ORACLE ANALYSIS WITHOUT V2 - FINAL SUMMARY")
print("="*80)

print(f"\n📊 KEY FINDINGS:")
print(f"   Training models: {len(model_names)} (excluded transformer_v2)")
print(f"   Training oracle F1: {training_oracle_no_v2['metrics']['f1']:.4f}")
print(f"   Training baseline F1: {training_baseline_no_v2['f1']:.4f}")

if test_oracle_no_v2 is not None:
    print(f"   Test models available: {len(available_test_models)} out of {len(model_names)}")
    print(f"   Test oracle F1: {test_oracle_no_v2['metrics']['f1']:.4f}")
    print(f"   Test baseline F1: {test_baseline_no_v2['f1']:.4f}")
    
    print(f"\n💡 INSIGHTS:")
    if test_oracle_no_v2['metrics']['f1'] > 0.85:
        print(f"   🎉 EXCELLENT performance even without V2!")
    elif test_oracle_no_v2['metrics']['f1'] > 0.80:
        print(f"   ✅ GOOD performance without V2")
    else:
        print(f"   ⚠️ Significant drop without V2 - it was crucial")
    
    if gap_without_v2 > 0.05:
        print(f"   🎯 Meta-learner could still be valuable without V2")
    else:
        print(f"   🤔 Limited meta-learning potential without V2")
    
    if meta_learner_f1 < baseline_without_v2:
        print(f"   🚨 MAJOR ISSUE: Meta-learner fails even without V2 dominance")
    
    print(f"\n🎯 RECOMMENDATIONS:")
    if meta_learner_f1 < baseline_without_v2:
        print(f"   1. 🔧 Fix fundamental meta-learner architecture issues")
        print(f"   2. 🎯 Try simpler model without meta-features first")
        print(f"   3. 📊 Use oracle ground truth instead of penalty-based")
        print(f"   4. 🔄 Consider training on subset of better models only")
    else:
        print(f"   1. ✅ Meta-learner architecture is sound")
        print(f"   2. 🎯 Focus on removing V2 penalty bias")
        print(f"   3. 📈 Small improvements should reach target")

print("="*80)
print("✅ Complete oracle analysis without V2 finished!")

# Save complete results
if test_oracle_no_v2 is not None:
    complete_results = {
        'training_oracle_f1_no_v2': float(training_oracle_no_v2['metrics']['f1']),
        'training_baseline_f1_no_v2': float(training_baseline_no_v2['f1']),
        'test_oracle_f1_no_v2': float(test_oracle_no_v2['metrics']['f1']),
        'test_baseline_f1_no_v2': float(test_baseline_no_v2['f1']),
        'models_without_v2': available_test_models,
        'test_models_available': len(available_test_models),
        'test_models_total': len(model_names),
        'meta_learning_potential_no_v2': float(gap_without_v2),
        'max_selection_percentage': float(max_selection_pct),
        'meta_learner_vs_baseline_gap': float(meta_learner_f1 - baseline_without_v2),
        'training_selection_distribution': {k: v for k, v in training_oracle_no_v2['selection_distribution'].items()},
        'test_selection_distribution': {k: v for k, v in test_oracle_no_v2['selection_distribution'].items()}
    }
    
    results_file = os.path.join(META_LEARNING_DIR, 'complete_oracle_analysis_no_v2.json')
    with open(results_file, 'w') as f:
        json.dump(complete_results, f, indent=2)
    
    print(f"💾 Complete results saved: {results_file}")


🔧 COMPLETE FIXED ORACLE ANALYSIS WITHOUT TRANSFORMER V2
🎯 Goal: Complete oracle analysis with ALL 8 models (5 ML + 3 Transformers)
🔧 Fix: Using reference code approach to load ML test predictions
📊 Analysis: Proper comparison without transformer_v2
🚫 Excluding models: ['transformer_v2']

1. Loading Training Predictions (Excluding V2)
----------------------------------------
   ✓ ml_aac: 42845 samples
   ✓ ml_binary: 42845 samples
   ✓ ml_dpc: 42845 samples
   ✓ ml_physicochemical: 42845 samples
   ✓ ml_tpc: 42845 samples
   ✓ transformer_v1: 42845 samples
   🚫 Skipping transformer_v2 (excluded)
   ✓ transformer_v3: 42845 samples
   ✓ transformer_v4: 42845 samples

📊 Training data without V2:
   Samples: 42845
   Models: 8 (was 9, now 8)
   Available models: ['ml_aac', 'ml_binary', 'ml_dpc', 'ml_physicochemical', 'ml_tpc', 'transformer_v1', 'transformer_v3', 'transformer_v4']

2. Oracle Selection - Training Data (Without V2)
----------------------------------------
🔮 Calculating oracle

In [11]:
# ============================================================================
# COMPLETE FIXED ORACLE ANALYSIS WITHOUT V2 + ML-ONLY COMPARISON
# Based on reference code, properly loading ALL test predictions
# ============================================================================

print("\n" + "="*80)
print("🔧 COMPLETE FIXED ORACLE ANALYSIS WITHOUT TRANSFORMER V2 + ML-ONLY")
print("="*80)
print("🎯 Goal: Complete oracle analysis with ALL 8 models (5 ML + 3 Transformers)")
print("🔧 Fix: Using reference code approach to load ML test predictions")
print("📊 Analysis: Proper comparison without transformer_v2 + ML-only analysis")
print("="*80)

import numpy as np
import pandas as pd
import os
import pickle
import sys
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                           f1_score, roc_auc_score, matthews_corrcoef)
import json

# ============================================================================
# CONFIGURATION
# ============================================================================

BASE_DIR = 'results/exp_3'
TRAINING_PRED_DIR = f'{BASE_DIR}/ensemble/training_predictions'
META_LEARNING_DIR = f'{BASE_DIR}/meta_learning'

os.makedirs(META_LEARNING_DIR, exist_ok=True)

# Model files (same as reference, but we'll exclude V2)
MODEL_FILES = {
    # ML Models
    'ml_aac': 'ml_aac_training_predictions.csv',
    'ml_binary': 'ml_binary_training_predictions.csv', 
    'ml_dpc': 'ml_dpc_training_predictions.csv',
    'ml_physicochemical': 'ml_physicochemical_training_predictions.csv',
    'ml_tpc': 'ml_tpc_training_predictions.csv',
    
    # Transformer Models
    'transformer_v1': 'transformer_transformer_v1_20250703_112558_training_predictions.csv',
    'transformer_v2': 'transformer_transformer_v2_20250704_102324_training_predictions.csv',  # Will exclude
    'transformer_v3': 'transformer_transformer_v3_20250709_175607_training_predictions.csv',
    'transformer_v4': 'transformer_transformer_v4_20250710_095513_training_predictions.csv'
}

EXCLUDED_MODELS = ['transformer_v2']
ML_MODELS = ['ml_aac', 'ml_binary', 'ml_dpc', 'ml_physicochemical', 'ml_tpc']
TRANSFORMER_MODELS = ['transformer_v1', 'transformer_v3', 'transformer_v4']

print(f"🚫 Excluding models: {EXCLUDED_MODELS}")
print(f"🔧 ML Models: {ML_MODELS}")
print(f"🤖 Transformer Models: {TRANSFORMER_MODELS}")

# ============================================================================
# 1. LOAD TRAINING PREDICTIONS (EXCLUDING V2)
# ============================================================================

print("\n1. Loading Training Predictions (Excluding V2)")
print("-" * 40)

training_predictions = {}
training_prediction_matrix = []
model_names = []
training_true_labels = None

for model_name, filename in MODEL_FILES.items():
    # Skip excluded models
    if model_name in EXCLUDED_MODELS:
        print(f"   🚫 Skipping {model_name} (excluded)")
        continue
        
    file_path = os.path.join(TRAINING_PRED_DIR, filename)
    
    if os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            print(f"   ✓ {model_name}: {len(df)} samples")
            
            training_predictions[model_name] = {
                'probabilities': df['probability'].values,
                'predictions': (df['probability'] > 0.5).astype(int),
                'true_labels': df['true_label'].values
            }
            
            training_prediction_matrix.append(df['probability'].values)
            model_names.append(model_name)
            
            if training_true_labels is None:
                training_true_labels = df['true_label'].values
                
        except Exception as e:
            print(f"   ❌ Error loading {model_name}: {e}")
    else:
        print(f"   ⚠️ File not found: {filename}")

training_prediction_matrix = np.column_stack(training_prediction_matrix)
print(f"\n📊 Training data without V2:")
print(f"   Samples: {len(training_true_labels)}")
print(f"   Models: {len(model_names)} (was 9, now {len(model_names)})")
print(f"   Available models: {model_names}")

# ============================================================================
# 2. TRAINING ORACLE ANALYSIS
# ============================================================================

print("\n2. Oracle Selection - Training Data (Without V2)")
print("-" * 40)

def calculate_oracle_performance(prediction_matrix, true_labels, model_names, data_type="Training"):
    """Calculate oracle (perfect selection) performance"""
    
    n_samples, n_models = prediction_matrix.shape
    oracle_predictions = []
    oracle_probabilities = []
    selected_models = []
    
    print(f"🔮 Calculating oracle selection for {n_samples} {data_type.lower()} samples...")
    print(f"   Using {n_models} models")
    
    for i in range(n_samples):
        true_label = true_labels[i]
        sample_predictions = prediction_matrix[i]
        
        if true_label == 1:
            best_model_idx = np.argmax(sample_predictions)
        else:
            best_model_idx = np.argmin(sample_predictions)
        
        oracle_probabilities.append(sample_predictions[best_model_idx])
        oracle_predictions.append(1 if sample_predictions[best_model_idx] > 0.5 else 0)
        selected_models.append(best_model_idx)
    
    oracle_predictions = np.array(oracle_predictions)
    oracle_probabilities = np.array(oracle_probabilities)
    selected_models = np.array(selected_models)
    
    metrics = {
        'accuracy': accuracy_score(true_labels, oracle_predictions),
        'precision': precision_score(true_labels, oracle_predictions, zero_division=0),
        'recall': recall_score(true_labels, oracle_predictions, zero_division=0),
        'f1': f1_score(true_labels, oracle_predictions, zero_division=0),
        'auc': roc_auc_score(true_labels, oracle_probabilities),
        'mcc': matthews_corrcoef(true_labels, oracle_predictions)
    }
    
    unique_models, counts = np.unique(selected_models, return_counts=True)
    selection_dist = {}
    for model_idx, count in zip(unique_models, counts):
        if model_idx < len(model_names):
            selection_dist[model_names[model_idx]] = {
                'count': int(count),
                'percentage': float(count / n_samples * 100)
            }
    
    return {
        'metrics': metrics,
        'oracle_predictions': oracle_predictions,
        'oracle_probabilities': oracle_probabilities,
        'selected_models': selected_models,
        'selection_distribution': selection_dist
    }

# Calculate training oracle
training_oracle_no_v2 = calculate_oracle_performance(
    training_prediction_matrix, 
    training_true_labels, 
    model_names, 
    "Training (No V2)"
)

print(f"🎯 Training Oracle Performance (WITHOUT V2):")
for metric, value in training_oracle_no_v2['metrics'].items():
    print(f"   {metric.upper()}: {value:.4f}")

# ============================================================================
# 3. LOAD TEST PREDICTIONS USING REFERENCE CODE APPROACH
# ============================================================================

print("\n3. Loading Test Predictions - Reference Code Approach")
print("-" * 40)

# Try to load progress tracker (exactly like reference code)
try:
    # First, try to create progress tracker
    sys.path.append('.')
    from Section0_code import ProgressTracker
    progress_tracker = ProgressTracker(BASE_DIR)
    print("✅ Progress tracker loaded successfully")
    
    # Load required checkpoints (exactly like Section 8)
    required_checkpoints = ['data_loading', 'data_splitting', 'ml_models_enhanced']
    loaded_data = {}

    print("📥 Loading core checkpoints...")
    for checkpoint_name in required_checkpoints:
        try:
            checkpoint_data = progress_tracker.resume_from_checkpoint(checkpoint_name)
            if checkpoint_data:
                loaded_data[checkpoint_name] = checkpoint_data
                print(f"   ✓ Loaded {checkpoint_name} checkpoint")
            else:
                print(f"   ⚠️ {checkpoint_name} checkpoint not found")
        except Exception as e:
            print(f"   ❌ Error loading {checkpoint_name}: {e}")
            
except Exception as e:
    print(f"⚠️ Could not load progress tracker: {e}")
    print("Will try manual loading...")
    
    # Manual loading approach
    loaded_data = {}
    checkpoints_dir = f'{BASE_DIR}/checkpoints'
    
    for checkpoint_name in ['data_loading', 'data_splitting', 'ml_models_enhanced']:
        checkpoint_file = os.path.join(checkpoints_dir, f'{checkpoint_name}.pkl')
        if os.path.exists(checkpoint_file):
            try:
                with open(checkpoint_file, 'rb') as f:
                    loaded_data[checkpoint_name] = pickle.load(f)
                print(f"   ✓ Manually loaded {checkpoint_name}")
            except Exception as e:
                print(f"   ❌ Error manually loading {checkpoint_name}: {e}")

# Extract test data
test_true_labels = None
if 'data_loading' in loaded_data and 'data_splitting' in loaded_data:
    df_final = loaded_data['data_loading']['df_final']
    test_indices = loaded_data['data_splitting']['test_indices']
    
    test_df = df_final.iloc[test_indices]
    test_true_labels = test_df['target'].values
    
    print(f"✅ Test data extracted:")
    print(f"   Test samples: {len(test_indices)}")
    print(f"   Test labels - Positive: {sum(test_true_labels)}, Negative: {len(test_true_labels) - sum(test_true_labels)}")

# ============================================================================
# 4. LOAD ML TEST PREDICTIONS (REFERENCE CODE APPROACH)
# ============================================================================

print("\n4. Loading ML Test Predictions from Checkpoint")
print("-" * 40)

test_predictions = {}

# Load ML test predictions from checkpoint (exactly like reference code)
if 'ml_models_enhanced' in loaded_data:
    ml_data = loaded_data['ml_models_enhanced']
    
    if 'test_predictions' in ml_data:
        ml_test_preds = ml_data['test_predictions']
        print(f"   ✅ Found ML test predictions for: {list(ml_test_preds.keys())}")
        
        for feature_type, pred_data in ml_test_preds.items():
            model_name = f'ml_{feature_type}'
            
            # Skip excluded models
            if model_name in EXCLUDED_MODELS:
                print(f"   🚫 Skipping {model_name} (excluded)")
                continue
            
            if 'pred_proba' in pred_data:
                test_predictions[model_name] = pred_data['pred_proba']
                print(f"   ✓ {model_name}: {len(pred_data['pred_proba'])} test samples")
    else:
        print("   ⚠️ No 'test_predictions' key found in ml_models_enhanced")
else:
    print("   ⚠️ ml_models_enhanced checkpoint not available")

# ============================================================================
# 5. LOAD TRANSFORMER TEST PREDICTIONS (REFERENCE CODE APPROACH)
# ============================================================================

print("\n5. Loading Transformer Test Predictions")
print("-" * 40)

# Load transformer test predictions (exactly like reference code)
transformer_dir = f'{BASE_DIR}/transformers'
if os.path.exists(transformer_dir):
    print(f"   📁 Searching in: {transformer_dir}")
    
    # Map model names to likely directory patterns (reference code approach)
    transformer_patterns = {
        'transformer_v1': ['transformer_v1_20250703_112558', 'transformer_v1'],
        'transformer_v2': ['transformer_v2_20250704_102324', 'transformer_v2'], 
        'transformer_v3': ['transformer_v3_20250709_175607', 'transformer_v3'],
        'transformer_v4': ['transformer_v4_20250710_095513', 'transformer_v4']
    }
    
    for model_name, patterns in transformer_patterns.items():
        # Skip excluded models
        if model_name in EXCLUDED_MODELS:
            print(f"   🚫 Skipping {model_name} (excluded)")
            continue
            
        found = False
        for pattern in patterns:
            for subdir in os.listdir(transformer_dir):
                if pattern in subdir:
                    subdir_path = os.path.join(transformer_dir, subdir)
                    if os.path.isdir(subdir_path):
                        pred_file = os.path.join(subdir_path, 'predictions', 'test_predictions.csv')
                        if os.path.exists(pred_file):
                            try:
                                df = pd.read_csv(pred_file)
                                if 'prediction_prob' in df.columns:
                                    test_predictions[model_name] = df['prediction_prob'].values
                                    print(f"   ✓ {model_name}: {len(df)} test samples")
                                    found = True
                                    break
                                elif 'probability' in df.columns:
                                    test_predictions[model_name] = df['probability'].values
                                    print(f"   ✓ {model_name}: {len(df)} test samples")
                                    found = True
                                    break
                            except Exception as e:
                                print(f"   ❌ Error loading {model_name}: {e}")
                if found:
                    break
            if found:
                break

# ============================================================================
# 6. CREATE COMPLETE TEST PREDICTION MATRIX
# ============================================================================

print("\n6. Creating Complete Test Prediction Matrix (Without V2)")
print("-" * 40)

# Create test prediction matrix in same order as training
test_prediction_matrix = []
available_test_models = []

print(f"📊 Combining test predictions in training order:")
for model_name in model_names:  # Use training model order (already excludes V2)
    if model_name in test_predictions:
        test_prediction_matrix.append(test_predictions[model_name])
        available_test_models.append(model_name)
        print(f"   ✓ {model_name}: Available for oracle analysis")
    else:
        print(f"   ❌ {model_name}: Missing test predictions")

if len(test_prediction_matrix) > 0:
    test_prediction_matrix = np.column_stack(test_prediction_matrix)
    
    print(f"\n📊 COMPLETE test data without V2:")
    print(f"   Available models: {len(available_test_models)} out of {len(model_names)}")
    print(f"   Prediction matrix shape: {test_prediction_matrix.shape}")
    print(f"   Models included: {available_test_models}")
    
    if test_true_labels is not None:
        print(f"   Test samples: {len(test_true_labels)}")
        
        # Verify alignment
        if len(test_true_labels) != test_prediction_matrix.shape[0]:
            min_len = min(len(test_true_labels), test_prediction_matrix.shape[0])
            test_true_labels = test_true_labels[:min_len]
            test_prediction_matrix = test_prediction_matrix[:min_len]
            print(f"   🔧 Truncated both to {min_len} samples for alignment")
        
        # Count ML vs Transformer models in test
        test_ml_count = len([m for m in available_test_models if m.startswith('ml_')])
        test_transformer_count = len([m for m in available_test_models if m.startswith('transformer_')])
        print(f"   📊 Test models: {test_ml_count} ML + {test_transformer_count} Transformers")
        
        # Calculate complete test oracle
        test_oracle_no_v2 = calculate_oracle_performance(
            test_prediction_matrix, 
            test_true_labels, 
            available_test_models, 
            "Test (No V2 - Complete)"
        )
        
        print(f"\n🎯 COMPLETE Test Oracle Performance (WITHOUT V2):")
        for metric, value in test_oracle_no_v2['metrics'].items():
            print(f"   {metric.upper()}: {value:.4f}")
    else:
        print("   ❌ No test labels available")
        test_oracle_no_v2 = None
else:
    print("   ❌ No test predictions available")
    test_oracle_no_v2 = None

# ============================================================================
# 7. ML-ONLY ANALYSIS
# ============================================================================

print("\n7. ML-Only Oracle Analysis")
print("=" * 40)

def create_ml_only_matrices(prediction_matrix, model_names, ml_models):
    """Extract only ML model predictions from the full matrix"""
    ml_indices = [i for i, name in enumerate(model_names) if name in ml_models]
    ml_matrix = prediction_matrix[:, ml_indices]
    ml_names = [model_names[i] for i in ml_indices]
    return ml_matrix, ml_names

# Training ML-only analysis
training_ml_matrix, training_ml_names = create_ml_only_matrices(
    training_prediction_matrix, model_names, ML_MODELS
)

print(f"🔧 Training ML-only analysis:")
print(f"   ML models available: {len(training_ml_names)} out of {len(ML_MODELS)}")
print(f"   ML models: {training_ml_names}")

if len(training_ml_names) > 0:
    training_oracle_ml_only = calculate_oracle_performance(
        training_ml_matrix,
        training_true_labels,
        training_ml_names,
        "Training (ML-Only)"
    )
    
    print(f"\n🎯 Training Oracle Performance (ML-ONLY):")
    for metric, value in training_oracle_ml_only['metrics'].items():
        print(f"   {metric.upper()}: {value:.4f}")

# Test ML-only analysis
if test_oracle_no_v2 is not None:
    test_ml_matrix, test_ml_names = create_ml_only_matrices(
        test_prediction_matrix, available_test_models, ML_MODELS
    )
    
    print(f"\n🔧 Test ML-only analysis:")
    print(f"   ML models available: {len(test_ml_names)} out of {len(ML_MODELS)}")
    print(f"   ML models: {test_ml_names}")
    
    if len(test_ml_names) > 0:
        test_oracle_ml_only = calculate_oracle_performance(
            test_ml_matrix,
            test_true_labels,
            test_ml_names,
            "Test (ML-Only)"
        )
        
        print(f"\n🎯 Test Oracle Performance (ML-ONLY):")
        for metric, value in test_oracle_ml_only['metrics'].items():
            print(f"   {metric.upper()}: {value:.4f}")
    else:
        test_oracle_ml_only = None
else:
    test_oracle_ml_only = None

# ============================================================================
# 8. TRANSFORMER-ONLY ANALYSIS
# ============================================================================

print("\n8. Transformer-Only Oracle Analysis")
print("=" * 40)

def create_transformer_only_matrices(prediction_matrix, model_names, transformer_models):
    """Extract only Transformer model predictions from the full matrix"""
    transformer_indices = [i for i, name in enumerate(model_names) if name in transformer_models]
    transformer_matrix = prediction_matrix[:, transformer_indices]
    transformer_names = [model_names[i] for i in transformer_indices]
    return transformer_matrix, transformer_names

# Training Transformer-only analysis
training_transformer_matrix, training_transformer_names = create_transformer_only_matrices(
    training_prediction_matrix, model_names, TRANSFORMER_MODELS
)

print(f"🤖 Training Transformer-only analysis:")
print(f"   Transformer models available: {len(training_transformer_names)} out of {len(TRANSFORMER_MODELS)}")
print(f"   Transformer models: {training_transformer_names}")

if len(training_transformer_names) > 0:
    training_oracle_transformer_only = calculate_oracle_performance(
        training_transformer_matrix,
        training_true_labels,
        training_transformer_names,
        "Training (Transformer-Only)"
    )
    
    print(f"\n🎯 Training Oracle Performance (TRANSFORMER-ONLY):")
    for metric, value in training_oracle_transformer_only['metrics'].items():
        print(f"   {metric.upper()}: {value:.4f}")

# Test Transformer-only analysis
if test_oracle_no_v2 is not None:
    test_transformer_matrix, test_transformer_names = create_transformer_only_matrices(
        test_prediction_matrix, available_test_models, TRANSFORMER_MODELS
    )
    
    print(f"\n🤖 Test Transformer-only analysis:")
    print(f"   Transformer models available: {len(test_transformer_names)} out of {len(TRANSFORMER_MODELS)}")
    print(f"   Transformer models: {test_transformer_names}")
    
    if len(test_transformer_names) > 0:
        test_oracle_transformer_only = calculate_oracle_performance(
            test_transformer_matrix,
            test_true_labels,
            test_transformer_names,
            "Test (Transformer-Only)"
        )
        
        print(f"\n🎯 Test Oracle Performance (TRANSFORMER-ONLY):")
        for metric, value in test_oracle_transformer_only['metrics'].items():
            print(f"   {metric.upper()}: {value:.4f}")
    else:
        test_oracle_transformer_only = None
else:
    test_oracle_transformer_only = None

# ============================================================================
# 9. COMPLETE BASELINE COMPARISON
# ============================================================================

print("\n9. Complete Baseline Comparison")
print("-" * 40)

def calculate_ensemble_baseline(prediction_matrix, true_labels):
    """Calculate simple average ensemble performance"""
    avg_predictions = np.mean(prediction_matrix, axis=1)
    avg_binary = (avg_predictions > 0.5).astype(int)
    
    return {
        'accuracy': accuracy_score(true_labels, avg_binary),
        'precision': precision_score(true_labels, avg_binary, zero_division=0),
        'recall': recall_score(true_labels, avg_binary, zero_division=0),
        'f1': f1_score(true_labels, avg_binary, zero_division=0),
        'auc': roc_auc_score(true_labels, avg_predictions),
        'mcc': matthews_corrcoef(true_labels, avg_binary)
    }

# Training baselines
training_baseline_no_v2 = calculate_ensemble_baseline(training_prediction_matrix, training_true_labels)
if len(training_ml_names) > 0:
    training_baseline_ml_only = calculate_ensemble_baseline(training_ml_matrix, training_true_labels)
if len(training_transformer_names) > 0:
    training_baseline_transformer_only = calculate_ensemble_baseline(training_transformer_matrix, training_true_labels)

print(f"📊 TRAINING SET COMPARISON:")
print(f"   🔧 ML-Only ({len(training_ml_names)} models):")
if len(training_ml_names) > 0:
    print(f"     Baseline F1: {training_baseline_ml_only['f1']:.4f}")
    print(f"     Oracle F1:   {training_oracle_ml_only['metrics']['f1']:.4f}")
    print(f"     Gap:         {training_oracle_ml_only['metrics']['f1'] - training_baseline_ml_only['f1']:.4f}")

print(f"\n   🤖 Transformer-Only ({len(training_transformer_names)} models):")
if len(training_transformer_names) > 0:
    print(f"     Baseline F1: {training_baseline_transformer_only['f1']:.4f}")
    print(f"     Oracle F1:   {training_oracle_transformer_only['metrics']['f1']:.4f}")
    print(f"     Gap:         {training_oracle_transformer_only['metrics']['f1'] - training_baseline_transformer_only['f1']:.4f}")

print(f"\n   🔄 All Models No V2 ({len(model_names)} models):")
print(f"     Baseline F1: {training_baseline_no_v2['f1']:.4f}")
print(f"     Oracle F1:   {training_oracle_no_v2['metrics']['f1']:.4f}")
print(f"     Gap:         {training_oracle_no_v2['metrics']['f1'] - training_baseline_no_v2['f1']:.4f}")

# Test baselines
if test_oracle_no_v2 is not None:
    test_baseline_no_v2 = calculate_ensemble_baseline(test_prediction_matrix, test_true_labels)
    if test_oracle_ml_only is not None:
        test_baseline_ml_only = calculate_ensemble_baseline(test_ml_matrix, test_true_labels)
    if test_oracle_transformer_only is not None:
        test_baseline_transformer_only = calculate_ensemble_baseline(test_transformer_matrix, test_true_labels)
    
    print(f"\n📊 TEST SET COMPARISON:")
    print(f"   🔧 ML-Only ({len(test_ml_names)} models):")
    if test_oracle_ml_only is not None:
        print(f"     Baseline F1: {test_baseline_ml_only['f1']:.4f}")
        print(f"     Oracle F1:   {test_oracle_ml_only['metrics']['f1']:.4f}")
        print(f"     Gap:         {test_oracle_ml_only['metrics']['f1'] - test_baseline_ml_only['f1']:.4f}")
    
    print(f"\n   🤖 Transformer-Only ({len(test_transformer_names)} models):")
    if test_oracle_transformer_only is not None:
        print(f"     Baseline F1: {test_baseline_transformer_only['f1']:.4f}")
        print(f"     Oracle F1:   {test_oracle_transformer_only['metrics']['f1']:.4f}")
        print(f"     Gap:         {test_oracle_transformer_only['metrics']['f1'] - test_baseline_transformer_only['f1']:.4f}")
    
    print(f"\n   🔄 All Models No V2 ({len(available_test_models)} models):")
    print(f"     Baseline F1: {test_baseline_no_v2['f1']:.4f}")
    print(f"     Oracle F1:   {test_oracle_no_v2['metrics']['f1']:.4f}")
    print(f"     Gap:         {test_oracle_no_v2['metrics']['f1'] - test_baseline_no_v2['f1']:.4f}")

# ============================================================================
# 10. DETAILED MODEL CONTRIBUTION ANALYSIS
# ============================================================================

print("\n10. Model Contribution Analysis")
print("=" * 40)

# Training analysis
print("🔍 Training Set - Model Selection Distribution:")
print(f"   🔄 All Models (No V2):")
for model_name, stats in training_oracle_no_v2['selection_distribution'].items():
    model_type = "🔧 ML" if model_name.startswith('ml_') else "🤖 Transformer"
    print(f"     {model_name} ({model_type}): {stats['count']} times ({stats['percentage']:.1f}%)")

if len(training_ml_names) > 0:
    print(f"\n   🔧 ML-Only:")
    for model_name, stats in training_oracle_ml_only['selection_distribution'].items():
        print(f"     {model_name}: {stats['count']} times ({stats['percentage']:.1f}%)")

if len(training_transformer_names) > 0:
    print(f"\n   🤖 Transformer-Only:")
    for model_name, stats in training_oracle_transformer_only['selection_distribution'].items():
        print(f"     {model_name}: {stats['count']} times ({stats['percentage']:.1f}%)")

# Test analysis
if test_oracle_no_v2 is not None:
    print(f"\n🔍 Test Set - Model Selection Distribution:")
    print(f"   🔄 All Models (No V2):")
    for model_name, stats in test_oracle_no_v2['selection_distribution'].items():
        model_type = "🔧 ML" if model_name.startswith('ml_') else "🤖 Transformer"
        print(f"     {model_name} ({model_type}): {stats['count']} times ({stats['percentage']:.1f}%)")
    
    if test_oracle_ml_only is not None:
        print(f"\n   🔧 ML-Only:")
        for model_name, stats in test_oracle_ml_only['selection_distribution'].items():
            print(f"     {model_name}: {stats['count']} times ({stats['percentage']:.1f}%)")
    
    if test_oracle_transformer_only is not None:
        print(f"\n   🤖 Transformer-Only:")
        for model_name, stats in test_oracle_transformer_only['selection_distribution'].items():
            print(f"     {model_name}: {stats['count']} times ({stats['percentage']:.1f}%)")

# ============================================================================
# 11. COMPARISON WITH ORIGINAL RESULTS
# ============================================================================

print("\n11. Comprehensive Comparison Analysis")
print("=" * 40)

print(f"🔄 TRAINING SET PERFORMANCE COMPARISON:")
print(f"   📊 WITH V2 (9 models):      Oracle F1: 0.9948")
print(f"   📊 WITHOUT V2 ({len(model_names)} models):   Oracle F1: {training_oracle_no_v2['metrics']['f1']:.4f}")
if len(training_ml_names) > 0:
    print(f"   🔧 ML-ONLY ({len(training_ml_names)} models):      Oracle F1: {training_oracle_ml_only['metrics']['f1']:.4f}")
if len(training_transformer_names) > 0:
    print(f"   🤖 TRANSFORMER-ONLY ({len(training_transformer_names)} models): Oracle F1: {training_oracle_transformer_only['metrics']['f1']:.4f}")

if test_oracle_no_v2 is not None:
    print(f"\n🔄 TEST SET PERFORMANCE COMPARISON:")
    print(f"   📊 WITH V2 (9 models):      Oracle F1: 0.9530")
    print(f"   📊 WITHOUT V2 ({len(available_test_models)} models):   Oracle F1: {test_oracle_no_v2['metrics']['f1']:.4f}")
    if test_oracle_ml_only is not None:
        print(f"   🔧 ML-ONLY ({len(test_ml_names)} models):      Oracle F1: {test_oracle_ml_only['metrics']['f1']:.4f}")
    if test_oracle_transformer_only is not None:
        print(f"   🤖 TRANSFORMER-ONLY ({len(test_transformer_names)} models): Oracle F1: {test_oracle_transformer_only['metrics']['f1']:.4f}")
    
    # Calculate value-add analysis
    print(f"\n💡 VALUE-ADD ANALYSIS (Test Set):")
    if test_oracle_ml_only is not None and test_oracle_transformer_only is not None:
        ml_only_f1 = test_oracle_ml_only['metrics']['f1']
        transformer_only_f1 = test_oracle_transformer_only['metrics']['f1']
        combined_f1 = test_oracle_no_v2['metrics']['f1']
        
        print(f"   🔧 ML-Only Oracle:          {ml_only_f1:.4f}")
        print(f"   🤖 Transformer-Only Oracle: {transformer_only_f1:.4f}")
        print(f"   🔄 Combined Oracle:         {combined_f1:.4f}")
        
        # Calculate improvements
        ml_over_transformer = ml_only_f1 - transformer_only_f1
        transformer_over_ml = transformer_only_f1 - ml_only_f1
        combined_over_best = combined_f1 - max(ml_only_f1, transformer_only_f1)
        
        print(f"\n   📈 Performance Gaps:")
        if ml_over_transformer > 0:
            print(f"     ML advantage over Transformers: +{ml_over_transformer:.4f} F1")
        else:
            print(f"     Transformer advantage over ML: +{abs(ml_over_transformer):.4f} F1")
        
        print(f"     Combined advantage over best: +{combined_over_best:.4f} F1")
        
        if combined_over_best > 0.01:
            print(f"   🎯 STRONG synergy between ML and Transformers!")
        elif combined_over_best > 0.005:
            print(f"   ✅ Moderate synergy between model types")
        else:
            print(f"   ⚠️ Limited synergy - one type dominates")

# ============================================================================
# 12. META-LEARNER IMPLICATIONS (ENHANCED)
# ============================================================================

print("\n12. Enhanced Meta-Learner Implications")
print("=" * 40)

if test_oracle_no_v2 is not None:
    meta_learner_f1 = 0.7655  # From your results
    baseline_with_v2 = 0.8131
    
    print(f"💡 META-LEARNER INSIGHTS (Enhanced):")
    
    # Compare meta-learner against all scenarios
    scenarios = {
        "Meta-learner (with V2)": meta_learner_f1,
        "Baseline (with V2)": baseline_with_v2,
        "Baseline (without V2)": test_baseline_no_v2['f1'],
        "Oracle (without V2)": test_oracle_no_v2['metrics']['f1'],
        "Oracle (with V2)": 0.9530
    }
    
    if test_oracle_ml_only is not None:
        scenarios["ML-Only Oracle"] = test_oracle_ml_only['metrics']['f1']
        scenarios["ML-Only Baseline"] = test_baseline_ml_only['f1']
    
    if test_oracle_transformer_only is not None:
        scenarios["Transformer-Only Oracle"] = test_oracle_transformer_only['metrics']['f1']
        scenarios["Transformer-Only Baseline"] = test_baseline_transformer_only['f1']
    
    print(f"\n📊 PERFORMANCE LADDER (Complete):")
    sorted_scenarios = sorted(scenarios.items(), key=lambda x: x[1], reverse=True)
    for i, (name, f1) in enumerate(sorted_scenarios, 1):
        if "Meta-learner" in name:
            marker = "🤖"
        elif "Oracle" in name:
            marker = "🎯"
        elif "Baseline" in name:
            marker = "📊"
        else:
            marker = "  "
        print(f"   {i:2d}. {marker} {name}: {f1:.4f}")
    
    # Find where meta-learner ranks
    meta_rank = next(i for i, (name, _) in enumerate(sorted_scenarios, 1) if "Meta-learner" in name)
    print(f"\n🤖 Meta-learner ranks {meta_rank} out of {len(scenarios)} scenarios")
    
    # Compare against different baselines
    gaps_analysis = {}
    if test_oracle_ml_only is not None:
        gaps_analysis["vs ML-Only Baseline"] = meta_learner_f1 - test_baseline_ml_only['f1']
    if test_oracle_transformer_only is not None:
        gaps_analysis["vs Transformer-Only Baseline"] = meta_learner_f1 - test_baseline_transformer_only['f1']
    gaps_analysis["vs Combined Baseline (no V2)"] = meta_learner_f1 - test_baseline_no_v2['f1']
    gaps_analysis["vs Combined Baseline (with V2)"] = meta_learner_f1 - baseline_with_v2
    
    print(f"\n📈 Meta-learner Performance Gaps:")
    for comparison, gap in gaps_analysis.items():
        status = "✅ BEATS" if gap > 0 else "❌ LOSES"
        print(f"   {comparison}: {gap:+.4f} ({status})")
    
    # Actionable recommendations
    print(f"\n🎯 ACTIONABLE RECOMMENDATIONS:")
    
    # Check if meta-learner beats any baseline
    beats_any_baseline = any(gap > 0 for gap in gaps_analysis.values())
    
    if not beats_any_baseline:
        print(f"   🚨 CRITICAL: Meta-learner fails against ALL baselines!")
        print(f"   1. 🔧 Complete architecture overhaul needed")
        print(f"   2. 🎯 Start with simplest possible meta-learner")
        print(f"   3. 📊 Train on oracle selections directly")
        
        # Check if ML-only would be better target
        if test_oracle_ml_only is not None:
            ml_gap = test_oracle_ml_only['metrics']['f1'] - test_baseline_ml_only['f1']
            combined_gap = test_oracle_no_v2['metrics']['f1'] - test_baseline_no_v2['f1']
            
            if ml_gap > combined_gap * 0.8:  # ML has >80% of combined potential
                print(f"   4. 💡 Consider ML-only meta-learner first (gap: {ml_gap:.4f})")
    else:
        # Find which baseline it beats
        best_beaten = max(gaps_analysis.items(), key=lambda x: x[1])
        print(f"   ✅ Best performance: {best_beaten[0]} (+{best_beaten[1]:.4f})")
        print(f"   1. 🔧 Build on this success - why does it work here?")
        print(f"   2. 📈 Target next weakest baseline")

# ============================================================================
# 13. FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("🔮 COMPLETE ORACLE ANALYSIS WITH ML-ONLY COMPARISON - FINAL SUMMARY")
print("="*80)

print(f"\n📊 MODEL COMPOSITION:")
print(f"   🔧 ML Models: {len([m for m in model_names if m.startswith('ml_')])}")
print(f"   🤖 Transformer Models: {len([m for m in model_names if m.startswith('transformer_')])}")
print(f"   🚫 Excluded: {EXCLUDED_MODELS}")

print(f"\n📊 TRAINING PERFORMANCE:")
print(f"   🔧 ML-Only Oracle F1:      {training_oracle_ml_only['metrics']['f1']:.4f}" if len(training_ml_names) > 0 else "   🔧 ML-Only: No models available")
print(f"   🤖 Transformer-Only F1:    {training_oracle_transformer_only['metrics']['f1']:.4f}" if len(training_transformer_names) > 0 else "   🤖 Transformer-Only: No models available")
print(f"   🔄 Combined (No V2) F1:    {training_oracle_no_v2['metrics']['f1']:.4f}")

if test_oracle_no_v2 is not None:
    print(f"\n📊 TEST PERFORMANCE:")
    print(f"   🔧 ML-Only Oracle F1:      {test_oracle_ml_only['metrics']['f1']:.4f}" if test_oracle_ml_only else "   🔧 ML-Only: No models available")
    print(f"   🤖 Transformer-Only F1:    {test_oracle_transformer_only['metrics']['f1']:.4f}" if test_oracle_transformer_only else "   🤖 Transformer-Only: No models available")
    print(f"   🔄 Combined (No V2) F1:    {test_oracle_no_v2['metrics']['f1']:.4f}")
    print(f"   📈 Combined (With V2) F1:  0.9530")
    
    print(f"\n💡 KEY INSIGHTS:")
    
    # Model type performance
    if test_oracle_ml_only and test_oracle_transformer_only:
        ml_f1 = test_oracle_ml_only['metrics']['f1']
        transformer_f1 = test_oracle_transformer_only['metrics']['f1']
        
        if abs(ml_f1 - transformer_f1) < 0.02:
            print(f"   ⚖️  ML and Transformers perform similarly ({ml_f1:.3f} vs {transformer_f1:.3f})")
        elif ml_f1 > transformer_f1:
            print(f"   🔧 ML models outperform Transformers (+{ml_f1-transformer_f1:.3f} F1)")
        else:
            print(f"   🤖 Transformers outperform ML models (+{transformer_f1-ml_f1:.3f} F1)")
    
    # Synergy analysis
    if test_oracle_ml_only and test_oracle_transformer_only:
        best_individual = max(test_oracle_ml_only['metrics']['f1'], test_oracle_transformer_only['metrics']['f1'])
        combined_improvement = test_oracle_no_v2['metrics']['f1'] - best_individual
        
        if combined_improvement > 0.01:
            print(f"   🚀 Strong synergy: Combined +{combined_improvement:.3f} over best individual")
        elif combined_improvement > 0.005:
            print(f"   ✅ Moderate synergy: Combined +{combined_improvement:.3f} over best individual")
        else:
            print(f"   ⚠️  Limited synergy: Combined +{combined_improvement:.3f} over best individual")
    
    # Meta-learner status
    ml_baseline_gap = meta_learner_f1 - test_baseline_no_v2['f1']
    if ml_baseline_gap < 0:
        print(f"   🚨 Meta-learner fails basic test: {ml_baseline_gap:.3f} below simple average")
    else:
        print(f"   ✅ Meta-learner beats simple average: +{ml_baseline_gap:.3f}")

print("="*80)
print("✅ Complete oracle analysis with ML-only comparison finished!")

# Save enhanced results
if test_oracle_no_v2 is not None:
    enhanced_results = {
        # Original results
        'training_oracle_f1_no_v2': float(training_oracle_no_v2['metrics']['f1']),
        'training_baseline_f1_no_v2': float(training_baseline_no_v2['f1']),
        'test_oracle_f1_no_v2': float(test_oracle_no_v2['metrics']['f1']),
        'test_baseline_f1_no_v2': float(test_baseline_no_v2['f1']),
        
        # ML-only results
        'training_oracle_f1_ml_only': float(training_oracle_ml_only['metrics']['f1']) if len(training_ml_names) > 0 else None,
        'training_baseline_f1_ml_only': float(training_baseline_ml_only['f1']) if len(training_ml_names) > 0 else None,
        'test_oracle_f1_ml_only': float(test_oracle_ml_only['metrics']['f1']) if test_oracle_ml_only else None,
        'test_baseline_f1_ml_only': float(test_baseline_ml_only['f1']) if test_oracle_ml_only else None,
        
        # Transformer-only results
        'training_oracle_f1_transformer_only': float(training_oracle_transformer_only['metrics']['f1']) if len(training_transformer_names) > 0 else None,
        'training_baseline_f1_transformer_only': float(training_baseline_transformer_only['f1']) if len(training_transformer_names) > 0 else None,
        'test_oracle_f1_transformer_only': float(test_oracle_transformer_only['metrics']['f1']) if test_oracle_transformer_only else None,
        'test_baseline_f1_transformer_only': float(test_baseline_transformer_only['f1']) if test_oracle_transformer_only else None,
        
        # Model composition
        'ml_models_available': len([m for m in available_test_models if m.startswith('ml_')]),
        'transformer_models_available': len([m for m in available_test_models if m.startswith('transformer_')]),
        'total_models_no_v2': len(available_test_models),
        
        # Model lists
        'models_without_v2': available_test_models,
        'ml_models': [m for m in available_test_models if m.startswith('ml_')],
        'transformer_models': [m for m in available_test_models if m.startswith('transformer_')],
        
        # Selection distributions
        'training_selection_distribution_all': {k: v for k, v in training_oracle_no_v2['selection_distribution'].items()},
        'test_selection_distribution_all': {k: v for k, v in test_oracle_no_v2['selection_distribution'].items()},
        'training_selection_distribution_ml_only': {k: v for k, v in training_oracle_ml_only['selection_distribution'].items()} if len(training_ml_names) > 0 else None,
        'test_selection_distribution_ml_only': {k: v for k, v in test_oracle_ml_only['selection_distribution'].items()} if test_oracle_ml_only else None,
        'training_selection_distribution_transformer_only': {k: v for k, v in training_oracle_transformer_only['selection_distribution'].items()} if len(training_transformer_names) > 0 else None,
        'test_selection_distribution_transformer_only': {k: v for k, v in test_oracle_transformer_only['selection_distribution'].items()} if test_oracle_transformer_only else None,
        
        # Meta-learner analysis
        'meta_learner_f1': 0.7655,
        'meta_learner_vs_baseline_no_v2': float(meta_learner_f1 - test_baseline_no_v2['f1']),
        'meta_learner_vs_ml_baseline': float(meta_learner_f1 - test_baseline_ml_only['f1']) if test_oracle_ml_only else None,
        'meta_learner_vs_transformer_baseline': float(meta_learner_f1 - test_baseline_transformer_only['f1']) if test_oracle_transformer_only else None,
    }
    
    results_file = os.path.join(META_LEARNING_DIR, 'enhanced_oracle_analysis_ml_comparison.json')
    with open(results_file, 'w') as f:
        json.dump(enhanced_results, f, indent=2)
    
    print(f"💾 Enhanced results saved: {results_file}")


🔧 COMPLETE FIXED ORACLE ANALYSIS WITHOUT TRANSFORMER V2 + ML-ONLY
🎯 Goal: Complete oracle analysis with ALL 8 models (5 ML + 3 Transformers)
🔧 Fix: Using reference code approach to load ML test predictions
📊 Analysis: Proper comparison without transformer_v2 + ML-only analysis
🚫 Excluding models: ['transformer_v2']
🔧 ML Models: ['ml_aac', 'ml_binary', 'ml_dpc', 'ml_physicochemical', 'ml_tpc']
🤖 Transformer Models: ['transformer_v1', 'transformer_v3', 'transformer_v4']

1. Loading Training Predictions (Excluding V2)
----------------------------------------
   ✓ ml_aac: 42845 samples
   ✓ ml_binary: 42845 samples
   ✓ ml_dpc: 42845 samples
   ✓ ml_physicochemical: 42845 samples
   ✓ ml_tpc: 42845 samples
   ✓ transformer_v1: 42845 samples
   🚫 Skipping transformer_v2 (excluded)
   ✓ transformer_v3: 42845 samples
   ✓ transformer_v4: 42845 samples

📊 Training data without V2:
   Samples: 42845
   Models: 8 (was 9, now 8)
   Available models: ['ml_aac', 'ml_binary', 'ml_dpc', 'ml_physicoc